# LLM-as-verifier baseline (lv1) over the frozen pool (GPU)

A model judges each of the 803 pool candidates from the request, the input schema and a summary of what the code
changed (`src/igt/baselines/llm_verifier.py`); no contract. One call per candidate that ran (about 760).
Same model as Baseline 3 by default.

After the run: download `llm_verifier_bundle.zip`, unzip it at the project root (lands in
`results/stage1/extensions/llm_verifier_<size>_lv1/`), then `.venv/bin/python eval/ext_score.py` (recall, false
rejections, McNemar vs Baselines 2 and 3).

In [ ]:
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"; LOAD_IN_4BIT = True
BACKEND = "hf"                  # "hf" | "vllm" | "accept_all" (pipeline check only, NOT results)
LIMIT = None                    # e.g. 20 for a quick check
SIZE = MODEL_ID.split("-")[-2].lower().replace(".", "_")
SYSTEM = f"llm_verifier_{SIZE}_lv1"
OUT = f"/kaggle/working/results/stage1/extensions/{SYSTEM}"
CACHE = "/kaggle/working/llm_cache"
NEEDED = ["eval/run_llm_verifier.py", "src/igt/baselines/llm_verifier.py", "benchmark/pool/pool_v4.jsonl"]

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json, collections, zipfile, datetime
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
WORK = "/kaggle/working"
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox / verifier / tools (own folder) + bitsandbytes ----------------
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
if BACKEND == "vllm":
    subprocess.run(["pip", "install", "-q", "-r", "requirements-vllm.txt"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages: only a warning.
probe = ("import pandas, numpy, torch, transformers; print('pandas', pandas.__version__, '| numpy', numpy.__version__, "
         "'| torch', torch.__version__, '| GPU', torch.cuda.is_available())")
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "GPU True" in p.stdout, "environment check failed"

In [ ]:
def sh(cmd, log=None):
    """Run a step, stream its output live, keep the full log."""
    t0 = time.time()
    p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if "Loading weights" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        open(log, "a").write(f"$ {' '.join(cmd)}\n" + "".join(lines) + f"\n--- exit {rc}, {(time.time() - t0) / 60:.1f} min ---\n")
    assert rc == 0, f"failed: {' '.join(cmd)}"
    return (time.time() - t0) / 60

cmd = ["python", "eval/run_llm_verifier.py", "--backend", BACKEND, "--model", MODEL_ID, "--system", SYSTEM,
       "--out", OUT, "--cache-dir", CACHE]
cmd += ["--load-in-4bit"] if LOAD_IN_4BIT else []
cmd += ["--limit", str(LIMIT)] if LIMIT else []
print(f"done in {sh(cmd, f'{OUT}/run.log'):.1f} min")
open(f"{OUT}/git_commit.txt", "w").write(COMMIT)
print(json.load(open(f"{OUT}/run_info.json")))

In [ ]:
# ---------------- ONE zip with everything needed on the laptop (unzip at the project root) ----------------
BUNDLE = f"{WORK}/llm_verifier_bundle.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for base in [OUT]:
        for dp, _, fs in os.walk(base):
            for f in fs:
                full = os.path.join(dp, f)
                z.write(full, os.path.relpath(full, WORK))
    z.writestr("llm_verifier_bundle_info.json", json.dumps(dict(commit=COMMIT, created=datetime.datetime.now().isoformat(timespec="seconds")), indent=2))
print("download:", BUNDLE, f"({os.path.getsize(BUNDLE) / 1e6:.1f} MB)")